In [276]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

In [277]:
filename = "G:/neural_network_from_scratch/heart diseases/heart.csv"
df = pd.read_csv(filename)
df.head(1)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0


In [278]:
df_2 = df.copy()
inputs = df_2.drop(columns='HeartDisease',axis=1)

In [279]:
inputs['bphr_ratio'] = inputs['RestingBP']/inputs['MaxHR']
inputs.drop(columns=['RestingBP','Cholesterol','MaxHR','RestingECG'],inplace=True)
inputs.head(1)

,Age,Sex,ChestPainType,FastingBS,ExerciseAngina,Oldpeak,ST_Slope,bphr_ratio
0,40,M,ATA,0,N,0.0,Up,0.813953


In [280]:
categorical_cols = [
    'Sex',
    'ChestPainType',
    'ExerciseAngina',
    'ST_Slope'
]

df_encoded = pd.get_dummies(
    inputs,
    columns=categorical_cols,
    dtype=int
)

df_encoded.head()

,Age,FastingBS,Oldpeak,bphr_ratio,Sex_F,Sex_M,ChestPainType_ASY,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,ExerciseAngina_N,ExerciseAngina_Y,ST_Slope_Down,ST_Slope_Flat,ST_Slope_Up
0,40,0,0.0,0.813953,0,1,0,1,0,0,1,0,0,0,1
1,49,0,1.0,1.025641,1,0,0,0,1,0,1,0,0,1,0
2,37,0,0.0,1.326531,0,1,0,1,0,0,1,0,0,0,1
3,48,0,1.5,1.277778,1,0,1,0,0,0,0,1,0,1,0
4,54,0,0.0,1.229508,0,1,0,0,1,0,1,0,0,0,1


In [281]:
df_scaled = df_encoded.copy()

continuous_cols = ['Age', 'Oldpeak', 'bphr_ratio']

df_scaled[continuous_cols] = (
    df_scaled[continuous_cols] - df_scaled[continuous_cols].mean()
) / df_scaled[continuous_cols].std()

In [282]:
X = df_scaled
y = df['HeartDisease'].values.reshape(-1,1)

print(X.shape)
print(y.shape)

(918, 15)
(918, 1)


In [283]:
from sklearn.model_selection import train_test_split

In [284]:
X = df_encoded.copy()
y = df['HeartDisease'].values.reshape(-1, 1)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

continuous_cols = ['Age', 'Oldpeak', 'bphr_ratio']

mean = X_train[continuous_cols].mean()
std = X_train[continuous_cols].std()

X_train[continuous_cols] = (
    X_train[continuous_cols] - mean
) / std

X_val[continuous_cols] = (
    X_val[continuous_cols] - mean
) / std

In [285]:
X_train.shape

(734, 15)

In [286]:
class LayerDense:
    def __init__(self,inputs,neurons):
        self.weights = np.random.rand(inputs,neurons) * np.sqrt(1/2)
        self.biases = np.zeros((1,neurons))
    def forward(self,x):
        self.output = np.dot(x,self.weights) + self.biases
        return self.output

def Relu(z):
    return np.maximum(0,z)

def sigmoid(z):
    return 1/(1+np.exp(-z))

def loss_func(y,y_pred):
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1-epsilon)
    loss = -np.mean(y*np.log(y_pred)+(1-y)*np.log(1-y_pred))
    return loss

In [287]:
layer_1 = LayerDense(15,64)
layer_2 = LayerDense(64,1)

In [288]:
def update_parameters(dw1,db1,dw2,db2,lr):
    
    layer_1.weights -= lr * dw1
    layer_1.biases -= lr * db1
    layer_2.weights -= lr * dw2
    layer_2.biases -= lr * db2

In [289]:
best_loss = float('inf')

best_W1 = None
best_b1 = None
best_W2 = None
best_b2 = None

In [290]:
learning_rate = 0.007
epochs = 5000
loss_history = []
epoch_list = [n for n in range(1,epochs+1)]

for epoch in range(epochs):

    m = X_train.shape[0]

    z1 = layer_1.forward(X_train)
    a1 = Relu(z1)

    z2 = layer_2.forward(a1)
    a2 = sigmoid(z2)

    loss = loss_func(y_train,a2)
    loss_history.append(loss)

    if loss < best_loss:
        best_loss = loss
        best_W1 = layer_1.weights.copy()
        best_b1 = layer_1.biases.copy()
        best_W2 = layer_2.weights.copy()
        best_b2 = layer_2.biases.copy()

    dz2 = a2 - y_train
    dw2 = np.dot(a1.T,dz2)/m
    db2 = np.sum(dz2, axis=0, keepdims=True) / m
    da1 = np.dot(dz2,layer_2.weights.T)
    dz1 = da1*(z1>0)
    dw1 = np.dot(X_train.T,dz1)/m
    db1 = np.sum(dz1, axis=0, keepdims=True) / m

    update_parameters(dw1,db1,dw2,db2,learning_rate)

    if (epoch%500 == 0):
        print(f"epoch: {epoch}    loss: {loss}")

epoch: 0    loss: 9.781640796098749
epoch: 500    loss: 0.3953611488286646
epoch: 1000    loss: 0.3578725017056089
epoch: 1500    loss: 0.34786455804276145
epoch: 2000    loss: 0.34397816021434413
epoch: 2500    loss: 0.3418772753166023
epoch: 3000    loss: 0.3406251160344136
epoch: 3500    loss: 0.33975911547613913
epoch: 4000    loss: 0.33905114252506013
epoch: 4500    loss: 0.33852353293195353


In [291]:
best_loss

np.float64(0.3379664932656104)

In [292]:
z1_val = np.dot(X_val, best_W1) + best_b1
a1_val = Relu(z1_val)

z2_val = np.dot(a1_val, best_W2) + best_b2
a2_val = sigmoid(z2_val)

val_loss = loss_func(y_val, a2_val)

print("Best Training Loss:", best_loss)
print("Validation Loss:", val_loss)

Best Training Loss: 0.3379664932656104
Validation Loss: 0.3270655429894312


In [297]:
y_pred = (a2_val >= 0.5).astype(int)
accuracy = np.mean(y_pred == y_val) * 100
print("Validation Accuracy:", accuracy)

Validation Accuracy: 89.13043478260869


In [298]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_val, y_pred))
print(classification_report(y_val, y_pred))

[[72 10]
 [10 92]]
              precision    recall  f1-score   support

           0       0.88      0.88      0.88        82
           1       0.90      0.90      0.90       102

    accuracy                           0.89       184
   macro avg       0.89      0.89      0.89       184
weighted avg       0.89      0.89      0.89       184



In [303]:
np.savez(
    'model_v3_updated.npz',
    W1=best_W1,
    b1=best_b1,
    W2=best_W2,
    b2=best_b2,
    mean=mean.values,
    std=std.values,
    columns=X_train.columns.to_numpy()
)